In [ ]:
import numpy as np

rng = np.random.default_rng(1129)


class network(object):
    def __init__(self, shape: list, batch_size: int) -> None:
        """
        创建神经网络

        参数：
            shape:神经网络形状
            batch_size:批量
        """

        self.shape = shape
        self.layer_num = len(shape)
        self.batch_size = batch_size

        self.activation = [
            np.zeros((shape[l], batch_size)) for l in range(self.layer_num)
        ]

        self.weights = [
            rng.normal(0, 1, (shape[l + 1], shape[l]))
            for l in range(self.layer_num - 1)
        ]
        self.grad_w = [
            np.zeros((shape[l + 1], shape[l])) for l in range(self.layer_num - 1)
        ]
        self.biases = [
            rng.normal(0, 1, (shape[l + 1], 1)) for l in range(self.layer_num - 1)
        ]

        self.grad_b = [np.zeros((shape[l + 1], 1)) for l in range(self.layer_num - 1)]

        class cache(object):
            def __init__(self, net) -> None:
                self.z = [
                    np.zeros((shape[l], batch_size)) for l in range(net.layer_num - 1)
                ]
                self.d = [
                    np.zeros((shape[l], batch_size)) for l in range(net.layer_num - 1)
                ]
                pass

        self.cache = cache(self)

    def shape_dict(self, l):
        """
        形状字典

        参数：
            l:显示层数
        """
        return {
            f"activation[{l+1}]": np.shape(self.activation[l + 1]),
            f"weights[{l}]": np.shape(self.weights[l]),
            f"activation[{l}]": np.shape(self.activation[l]),
            f"biases[{l}]": np.shape(self.biases[l]),
            f"z[{l}]": np.shape(self.cache.z[l]),
            f"d[{l}]": np.shape(self.cache.d[l]),
            f"d[{l+1}]": np.shape(self.cache.d[l + 1]),
            f"weights[{l+1}]": np.shape(self.weights[l + 1]),
        }

    def rng_input(self, loc=0, scale=1):
        """
        初始化输入(正态分布随机)

        参数：
            loc:均值
            scale:标准差
        """
        self.activation[0] = rng.normal(loc, scale, np.shape(self.activation[0]))

    def rng_ground_truth(self, loc=0.5, scale=0.1):
        self.gt = rng.normal(loc, scale, np.shape(self.activation[-1]))
        pass

    def shape_test(self, l):
        """
        形状字典

        参数：
            l:显示层数
        """
        return {
            f"d[{-1}]": np.shape(self.cache.d[-1]),
            f"grad_w[{-1}]": np.shape(self.grad_w[-1]),
            f"activation[{-1}]": np.shape(self.activation[-1]),
            f"activation[{-2}]": np.shape(self.activation[-2]),
            f"weights[{-l+1}]": np.shape(self.weights[-l + 1]),
            f"d[{-l}]": np.shape(self.cache.d[-l]),
            f"grad_w[{-l}]": np.shape(self.grad_w[-1]),
            f"activation[{-l}]": np.shape(self.activation[-l]),
        }


def sigmoid(x):
    return 1 / (1 + np.exp(-x))


def d_sigmoid(value):
    """
    sigmoid导数

    参数：
        value:sigmoid处理后的值
    """
    return value * (1 - value)


def d_cost(output, ground_truth, batch_size):
    """
    cost的导数

    参数：
        output:输出层数组
        ground_truth:真实值
        batch_size:批量
    """
    return (output - ground_truth) / batch_size


class propagation(object):
    def __init__(self, eta, net: network) -> None:
        self.eta = eta
        self.net = net

    def forward_propagation(self):
        for l in range(self.net.layer_num - 1):
            self.net.cache.z[l] = (
                self.net.weights[l] @ self.net.activation[l] + self.net.biases[l]
            )
            self.net.activation[l + 1] = sigmoid(self.net.cache.z[l])
            self.net.cache.d[l] = d_sigmoid(self.net.activation[l + 1])

    def back_propagation(self):
        d_c = d_cost(self.net.activation[-1], self.net.gt, self.net.batch_size)
        dw = self.net.cache.d[-1] * d_c
        self.net.grad_w[-1] = dw @ self.net.activation[-2].T
        self.net.grad_b[-1] = dw.sum(axis=1, keepdims=True)
        for l in range(2, self.net.layer_num):
            dw = (self.net.weights[-l + 1].T @ dw) * self.net.cache.d[-l]
            self.net.grad_w[-l] = dw @ self.net.activation[-l - 1].T
            self.net.grad_b[-l] = dw.sum(axis=1, keepdims=True)

    def gradient_descent(self):
        for l in range(self.net.layer_num - 1):
            self.net.weights[l] += -self.eta * self.net.grad_w[l]
            self.net.biases[l] += -self.eta * self.net.grad_b[l]
        pass


batch_size = 3
shape = [784, 128, 64, 10]
eta = 0.05
times = 100000

net = network(shape, batch_size)
net.rng_input()
net.rng_ground_truth()
prop = propagation(eta, net)

for step in range(times):
    prop.forward_propagation()
    prop.back_propagation()
    prop.gradient_descent()


SyntaxError: invalid syntax (2050628367.py, line 169)